In [1]:
import torch
from ultralytics import YOLO
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms
from PIL import Image
import numpy as np

In [2]:
# Load your trained YOLOv8 model
yolo_model = YOLO("runs/detect/yolov8s_mapillary_subset_cpu10/weights/best.pt")


In [12]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.datasets import ImageFolder

train_dir = "crops_per_label"

dataset = ImageFolder(train_dir)
class_names = dataset.classes   # SORTED and consistent
num_classes = len(class_names)
class_names = dataset.classes   # SORTED and consistent

weights = EfficientNet_B0_Weights.DEFAULT
classifier = efficientnet_b0(weights=weights)
classifier.classifier[1] = torch.nn.Linear(
    classifier.classifier[1].in_features,
    num_classes
)

classifier.load_state_dict(
    torch.load("street_sign_classifier_weights.pth", map_location="cpu")
)
classifier.eval()

classifier_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


C:\Users\ralfm\AppData\Local\Temp\ipykernel_25864\910958862.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("street_sign_classifier_weights.pth", map_locatio

In [21]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import torch


def detect_and_classify(image_path, conf_threshold=0.3, show_image=False):
    """
    Input: full image path
    Output: list of dicts with bbox + predicted label
    """
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    # YOLO detection
    results = yolo_model(image_np, conf=conf_threshold)[0]

    outputs = []

    if results.boxes is None:
        return outputs

    for box in results.boxes:
        # Bounding box
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        det_conf = float(box.conf[0])

        # Crop
        crop = image.crop((x1, y1, x2, y2))

        # Classifier inference
        input_tensor = classifier_transform(crop).unsqueeze(0)

        with torch.no_grad():
            logits = classifier(input_tensor)
            probs = torch.softmax(logits, dim=1)
            cls_idx = probs.argmax(dim=1).item()
            cls_conf = probs[0, cls_idx].item()

        outputs.append({
            "bbox": [x1, y1, x2, y2],
            "det_conf": det_conf,
            "label": class_names[cls_idx],
            "cls_conf": cls_conf
        })

    # Visualization using PIL
    if show_image:
        draw = ImageDraw.Draw(image)

        try:
            font = ImageFont.truetype("arial.ttf", size=20)
        except IOError:
            font = ImageFont.load_default()

        for out in outputs:
            x1, y1, x2, y2 = out["bbox"]
            label = out["label"]
            score = out["cls_conf"]

            # Draw bounding box
            draw.rectangle(
                [(x1, y1), (x2, y2)],
                outline="red",
                width=3
            )

            text = f"{label} ({score:.2f})"

            # Text background
            text_bbox = draw.textbbox((x1, y1), text, font=font)
            draw.rectangle(text_bbox, fill="red")

            # Draw text
            draw.text(
                (x1, y1),
                text,
                fill="white",
                font=font
            )

        image.show()

    return outputs


In [23]:
image_path = "dataset/images/val/0ZZSKbvK16JFH90FSWT_Ww.jpg"

predictions = detect_and_classify(image_path, show_image=True)

for p in predictions:
    print(
        f"BBox: {p['bbox']}, "
        f"Label: {p['label']}, "
        f"Detection conf: {p['det_conf']:.2f}, "
        f"Class conf: {p['cls_conf']:.2f}"
    )



0: 448x640 1 streetsign, 82.7ms
Speed: 1.8ms preprocess, 82.7ms inference, 2.0ms postprocess per image at shape (1, 3, 448, 640)
BBox: [1692, 1158, 1806, 1268], Label: complementary--chevron-right--g5, Detection conf: 0.73, Class conf: 0.51


In [38]:
import glob
import random

VAL_DIR = "dataset/images/val"

# Collect all images
image_paths = glob.glob(f"{VAL_DIR}/*.jpg") + glob.glob(f"{VAL_DIR}/*.png")
assert len(image_paths) > 0, "No images found in val directory!"

# Pick one random image
image_path = random.choice(image_paths)

print(f"Selected image: {image_path}")

# Run pipeline
predictions = detect_and_classify(image_path, show_image=True)

# Print results
for p in predictions:
    print(
        f"BBox: {p['bbox']}, "
        f"Label: {p['label']}, "
        f"Detection conf: {p['det_conf']:.2f}, "
        f"Class conf: {p['cls_conf']:.2f}"
    )


Selected image: dataset/images/val\2Lm6GYOEavrV_NavP4ZJNA.jpg

0: 480x640 2 streetsigns, 82.6ms
Speed: 1.8ms preprocess, 82.6ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)
BBox: [1179, 1218, 1216, 1259], Label: other-sign, Detection conf: 0.71, Class conf: 0.99
BBox: [1929, 1209, 1970, 1247], Label: regulatory--no-stopping--g15, Detection conf: 0.38, Class conf: 0.33
